# NB_R01 · 시장 국면 피처 생성 (Gold → gold_regime_features)

**[시장국면분석ML] 1/3 단계.** 스트레스엔진ML과 같은 `gold_stress_daily` v1.1(Gold 인증 스냅샷)을 읽어 날짜 단위 시장 상태 피처를 만듭니다. Bronze·Silver·Gold는 새로 만들지 않고 그대로 씁니다.

- 입력: `Files/gold_evidence/gold_stock_market_daily_v11/` 의 `health.json`(ready 필수), `latest.json`, 인증서 해시가 맞는 `gold.parquet` + `input_eligibility.parquet`
- 처리: 수익률·이동창은 전체 거래일 달력에서 계산하고, 행별 사용 가능 여부(`input_eligible`)를 적용합니다(팀 Gold 인증서의 required_consumer_order). 사용 가능한 종목이 10개 미만인 날은 피처를 비웁니다.
- 출력: Delta `gold_regime_features`, `Files/regime/runs/<RUN_ID>/context.json`
- 기본 Lakehouse: **LH_MarketRegime** (팀 `lh_whatif`의 Gold는 읽기 전용 바로가기 `Files/gold_evidence`, `Tables/gold_stress_daily`로 읽고, 결과는 LH_MarketRegime에만 씀)

In [ ]:
RUN_ID = ""                 # 파이프라인에서 @pipeline().RunId 전달. 비우면 새로 만듭니다
USE_CERTIFIED_GOLD = True   # False면 Delta gold_stress_daily를 직접 읽음(quality_status='valid' 행만 사용)
GOLD_TABLE = "gold_stress_daily"
CONFIG_OVERRIDES_JSON = ""  # 예: '{"min_train_days": 750, "refit_every": 21}' 

In [ ]:
# 모듈 무결성 확인 후 불러오기 (팀 Gold 노트북과 같은 해시 확인 방식)
import sys, json, hashlib, importlib, time
from pathlib import Path
import numpy as np, pandas as pd
assert Path("/lakehouse/default/Files").is_dir(), "기본 Lakehouse LH_MarketRegime을 연결하세요."
# 팀 원천 Lakehouse(lh_whatif)에는 쓰지 않습니다. 기본 Lakehouse가 lh_whatif이면 여기서 멈춥니다.
try:
    import notebookutils
    _lh = (notebookutils.runtime.context.get("defaultLakehouseName") or "")
except Exception:
    _lh = ""
if _lh.lower() == "lh_whatif" or Path("/lakehouse/default/Files/bronze").exists():
    raise RuntimeError("TEAM_LAKEHOUSE_DETECTED: 기본 Lakehouse를 LH_MarketRegime으로 연결하세요.")
MODULE_DIR = Path("/lakehouse/default/Files/reference/market_regime/v1")
_manifest = json.loads((MODULE_DIR / "manifest.json").read_text(encoding="utf8"))
for _f, _h in _manifest["files"].items():
    if hashlib.sha256((MODULE_DIR / _f).read_bytes()).hexdigest() != _h:
        raise RuntimeError("REGIME_MODULE_HASH_MISMATCH:" + _f)
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import hmm_numpy, regime_core as rc, fabric_io as fio
for _m in (hmm_numpy, rc, fio):
    importlib.reload(_m)
try:
    spark.conf.set("spark.sql.session.timeZone", "UTC")   # 팀 Gold 노트북과 같은 기준
except Exception:
    pass
print("market_regime", rc.CORE_VERSION, rc.MODEL_VERSION)

In [ ]:
RUN_ID = RUN_ID or fio.new_run_id()
cfg = rc.make_config(json.loads(CONFIG_OVERRIDES_JSON) if CONFIG_OVERRIDES_JSON.strip() else None)
t0 = time.time()
gold, elig, gold_version = fio.load_gold(spark, USE_CERTIFIED_GOLD, GOLD_TABLE)
print("Gold:", gold.shape, json.dumps(gold_version, ensure_ascii=False, default=str))
feat, quality = rc.build_features(gold, elig, cfg)
print(f"피처 {feat.shape} | {quality['first_date']} → {quality['last_date']} | 사용 가능 {quality['dates_usable']}일, 제외 {quality['dates_excluded']}일 | {time.time()-t0:.1f}s")
print("종목별 첫 사용일:", {a["asset_id"]: a["first_usable_date"] for a in quality["assets"]})

In [ ]:
# 게시 전 검사: 마지막 피처 날짜 = Gold 최신일
last_ok = feat.index[feat["date_ok"]].max()
if gold_version.get("latest_gold_date") and str(last_ok.date()) != gold_version["latest_gold_date"]:
    raise RuntimeError(f"FEATURE_LATEST_DATE_MISMATCH: {last_ok.date()} != {gold_version['latest_gold_date']}")
need = cfg["min_train_days"] + 252
if int(feat["date_ok"].sum()) < need:
    raise RuntimeError(f"HISTORY_TOO_SHORT: 사용 가능 {int(feat['date_ok'].sum())}일 < 필요 {need}일")
missing_share = feat.loc[feat["date_ok"], cfg["hmm_features"]].isna().mean().round(4).to_dict()
print("HMM 피처 결측 비율(사용 가능일 기준):", missing_share)

In [ ]:
n = fio.save_table(spark, feat, "gold_regime_features")
ctx = {"run_id": RUN_ID, "created_at_utc": fio.now_utc().isoformat(), "config": cfg,
       "gold_version": gold_version, "quality": quality, "core_version": rc.CORE_VERSION,
       "module_manifest": _manifest}
fio.write_json_atomic(fio.run_dir(RUN_ID) / "context.json", ctx)
fio.write_json_atomic(fio.REGIME_ROOT / "runs/current_run.json", {"run_id": RUN_ID, "created_at_utc": ctx["created_at_utc"]})
print("gold_regime_features", n, "rows | RUN_ID", RUN_ID)
display(spark.table("gold_regime_features").orderBy("trading_date", ascending=False).limit(5))